# Attention sink visualization with 4 or 8 bit quantization for Llama in BertViz
Make sure that you can access the Llama models from huggingface:

https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

https://huggingface.co/meta-llama/Meta-Llama-3-8B

Afterwards copy the access token from the huggingface setting for the login.

In [9]:
pip install bitsandbytes accelerate transformers bertviz --quiet

^C
ERROR: Operation cancelled by user

[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `huggingface-cli whoami` to get more information or `huggingface-cli logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): Traceback (most recent call last):
  File "/Users/chunwei/PycharmProjec

In [ ]:
from transformers import AutoTokenizer, AutoModel, utils
#utils.logging.set_verbosity_error()  # Suppress standard warnings

# load the model
# meta-llama/Meta-Llama-3-8B
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# load the model in 4 bit
# The VRAM of google colab also fits 8 bit
model = AutoModel.from_pretrained(model_name, output_attentions=True, load_in_4bit=True, device_map="auto")

/Users/chunwei/PycharmProjects/attentivetrim/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [ ]:
input_sentence = """The sentence you are given might be too wordy, complicated, or unclear. Rewrite the sentence and make your writing clearer by keeping it concise. Whenever possible, break complex sentences into multiple sentences and eliminate unnecessary words.\n
Input: If you have any questions about my rate or if you find it necessary to increase or decrease the scope for this project, please let me know.\n\nOutput:"""
input_sentence = "Generate a positive review for a place."
inputs = tokenizer.encode(input_sentence, return_tensors='pt')
outputs = model(inputs)
attention = outputs[-1]  # Output includes attention weights when output_attentions=True
tokens = tokenizer.convert_ids_to_tokens(inputs[0])
# save the attention weights into a file
import pickle
with open('attention.pkl', 'wb') as f:
    pickle.dump(attention, f)
    
# save the tokens into a file
with open('tokens.pkl', 'wb') as f:
    pickle.dump(tokens, f)
    


In [ ]:
import pickle
from bertviz import head_view
from bertviz import model_view
# load the attention weights from the file
with open('../data/tensor/attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open('../data/tensor/tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")
html_head_view = head_view(attention, tokens)

In [ ]:
tokens

In [ ]:
model_view(attention, tokens)

# Further literature

A Primer on the Inner Workings of Transformer-based Language Models: https://arxiv.org/abs/2405.00208

Massive Activations in Large Language Models
https://arxiv.org/abs/2402.17762

Understanding and Minimising Outlier Features in Neural Network Training
https://arxiv.org/abs/2405.19279

signals exchanged in the tail end of the spectrum are responsible for
attention sinking
https://arxiv.org/abs/2402.09221

Information Flow Routes: Automatically Interpreting Language Models at Scale
https://arxiv.org/abs/2403.00824

MInference 1.0: Accelerating Pre-filling for Long-Context LLMs via Dynamic Sparse Attention https://huggingface.co/papers/2407.02490

Efficient implementations of state-of-the-art linear attention models in Pytorch and Triton https://github.com/sustcsonglin/flash-linear-attention

In [4]:
import pickle
from bertviz import head_view
from bertviz import model_view
i = 1
j = 1
# load the attention weights from the file
with open(f'../data/tensor/context{i}-question{j}_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open(f'../data/tensor/context{i}-question{j}_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")

Attention shape: 32
tokens length: 263


In [6]:

# save the complete model view
html_head_view = head_view(attention, tokens, html_action='return')
with open("all_head_view.html", 'w') as file:
    file.write(html_head_view.data)

html_model_view = model_view(attention, tokens, html_action='return')
with open("all_model_view.html", 'w') as file:
    file.write(html_model_view.data)

# save the view just for certain layers if the browser can't display the whole
# shorter inputs are easier to display
layers = [9]
html_head_view = head_view(attention, tokens, html_action='return', include_layers=layers)

with open("short_head_view.html", 'w') as file:
    file.write(html_head_view.data)

html_model_view = model_view(attention, tokens, html_action='return', include_layers=layers)
with open("short_model_view.html", 'w') as file:
    file.write(html_model_view.data)

In [ ]:
html_head_view

In [ ]:
html_model_view

In [ ]:
attention[0]

In [ ]:
attention[0].shape

In [ ]:
attention[0][0][0][110]

In [35]:
import torch
#3-15 grd for title
# -10: for question
topk = 30
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 15 and val >= 3:
                count += 1
    print(f"Layer: {layer} Count: {count}")

Layer: 0 Count: 8
Layer: 1 Count: 0
Layer: 2 Count: 20
Layer: 3 Count: 23
Layer: 4 Count: 10
Layer: 5 Count: 6
Layer: 6 Count: 10
Layer: 7 Count: 12
Layer: 8 Count: 18
Layer: 9 Count: 16
Layer: 10 Count: 29
Layer: 11 Count: 15
Layer: 12 Count: 5
Layer: 13 Count: 25
Layer: 14 Count: 29
Layer: 15 Count: 28
Layer: 16 Count: 28
Layer: 17 Count: 36
Layer: 18 Count: 33
Layer: 19 Count: 36
Layer: 20 Count: 36
Layer: 21 Count: 40
Layer: 22 Count: 38
Layer: 23 Count: 37
Layer: 24 Count: 37
Layer: 25 Count: 24
Layer: 26 Count: 13
Layer: 27 Count: 21
Layer: 28 Count: 19
Layer: 29 Count: 24
Layer: 30 Count: 31


In [37]:
import pickle
from bertviz import head_view
from bertviz import model_view
# load the attention weights from the file
with open('../data/tensor/abstract_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open('../data/tensor/abstract_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")

Attention shape: 32
tokens length: 112


In [38]:
import torch
#71-99 grd for abstract
# -10: for question
topk = 30
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 99 and val >= 71:
                count += 1
    print(f"Layer: {layer} Count: {count}")

Layer: 0 Count: 165
Layer: 1 Count: 154
Layer: 2 Count: 103
Layer: 3 Count: 139
Layer: 4 Count: 144
Layer: 5 Count: 133
Layer: 6 Count: 142
Layer: 7 Count: 119
Layer: 8 Count: 104
Layer: 9 Count: 103
Layer: 10 Count: 101
Layer: 11 Count: 102
Layer: 12 Count: 121
Layer: 13 Count: 88
Layer: 14 Count: 85
Layer: 15 Count: 88
Layer: 16 Count: 87
Layer: 17 Count: 91
Layer: 18 Count: 100
Layer: 19 Count: 92
Layer: 20 Count: 84
Layer: 21 Count: 91
Layer: 22 Count: 94
Layer: 23 Count: 88
Layer: 24 Count: 91
Layer: 25 Count: 99
Layer: 26 Count: 103
Layer: 27 Count: 92
Layer: 28 Count: 107
Layer: 29 Count: 103
Layer: 30 Count: 93


In [6]:
# return the top 10 values's indices
topk = torch.topk(sum[-10:], 20)
topk.indices


NameError: name 'torch' is not defined

In [5]:
tokens[71:104]

['@m',
 'icrosoft',
 '.com',
 'Ċ',
 'ĠĠĠĠĠĠĠĠĠĠĠ',
 'ĠThis',
 'Ġpaper',
 'Ġevaluates',
 'Ġthe',
 'Ġsuitability',
 'Ġof',
 'ĠApache',
 'ĠArrow',
 ',',
 'ĠPar',
 'quet',
 ',',
 'Ġand',
 'ĠOR',
 'C',
 'Ġas',
 'Ġformats',
 'Ġfor',
 'Ġsub',
 'sum',
 'ption',
 'Ġin',
 'Ġan',
 'Ġanalytical',
 'ĠDB',
 'MS',
 '.',
 'ĠĊĠĠĠĠĠĠĠĠĠĠĠĠĊ']

In [3]:
# print token with indices started from 0
for i in range(0, len(tokens)):
    print(f"{i}: {tokens[i]}")

0: <|begin_of_text|>
1: Context
2: :
3: ĠDetect
4: ing
5: ĠAI
6: -
7: Generated
8: ĠText
9: :
10: ĠFactors
11: ĠInflu
12: encing
13: Ċ
14: Detect
15: ability
16: Ġwith
17: ĠCurrent
18: ĠMethods
19: Ċ
20: K
21: ath
22: leen
23: ĠC
24: .
25: ĠĊ
26: ĠĠĠĠĠĠĠĠĠĠĠĠ
27: ĠFraser
28: Ċ
29: k
30: ath
31: leen
32: .fr
33: aser
34: @n
35: rc
36: -cn
37: rc
38: .gc
39: .ca
40: Ċ
41: Hillary
42: ĠDaw
43: kins
44: Ċ
45: hill
46: ary
47: .d
48: aw
49: kins
50: @n
51: rc
52: -cn
53: rc
54: .gc
55: .ca
56: Ċ
57: S
58: vet
59: l
60: ana
61: ĠĊ
62: ĠĠĠĠĠĠĠĠĠĠĠĠ
63: ĠKir
64: itchen
65: ko
66: Ċ
67: sv
68: et
69: l
70: ana
71: .k
72: ir
73: itchen
74: ko
75: @n
76: rc
77: -cn
78: rc
79: .gc
80: .ca
81: Ċ
82: National
83: ĠResearch
84: ĠCouncil
85: ĠCanada
86: Ċ
87: 120
88: 0
89: ĠMontreal
90: ĠRoad
91: ,
92: ĠĊ
93: ĠĠĠĠĠĠĠĠĠĠĠĠ
94: ĠOttawa
95: ,
96: ĠCanada
97: Ċ
98: Abstract
99: Ċ
100: Large
101: Ġlanguage
102: Ġmodels
103: Ġ(
104: LL
105: Ms
106: )
107: Ġhave
108: Ġadvanced
109: Ġto
110: Ġa
111: Ġpoint
11

In [1]:
# show the layer level attention
import pickle
for i in range(0, 3):
    for j in range(0, 2):
        print(f"context: {i} question: {j}")
        
        # load the attention weights from the file
        with open(f'../data/tensor/context{i}-question{j}_attention.pkl', 'rb') as f:
            attention = pickle.load(f)
            
        # load the tokens from the file
        with open(f'../data/tensor/context{i}-question{j}_tokens.pkl', 'rb') as f:
            tokens = pickle.load(f)
            
        print(f"Attention shape: {len(attention)}")
        print(f"tokens length: {len(tokens)}")

context: 0 question: 0
Attention shape: 32
tokens length: 114
context: 0 question: 1
Attention shape: 32
tokens length: 116
context: 1 question: 0
Attention shape: 32
tokens length: 261
context: 1 question: 1
Attention shape: 32
tokens length: 263
context: 2 question: 0
Attention shape: 32
tokens length: 272
context: 2 question: 1
Attention shape: 32
tokens length: 274


In [2]:
i = 0
j = 0
print(f"context: {i} question: {j}")
        
# load the attention weights from the file
with open(f'../data/tensor/context{i}-question{j}_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open(f'../data/tensor/context{i}-question{j}_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")
for i in range(0, len(tokens)):
    print(f"{i}: {tokens[i]}")

context: 0 question: 0
Attention shape: 32
tokens length: 114
0: <|begin_of_text|>
1: Context
2: :
3: ĠA
4: ĠDeep
5: ĠDive
6: Ġinto
7: ĠCommon
8: ĠOpen
9: ĠFormats
10: Ġfor
11: ĠAnaly
12: tical
13: ĠDB
14: MS
15: s
16: Ċ
17: ĠĠĠĠĠĠĠĠĠĠĠ
18: ĠChun
19: wei
20: ĠLiu
21: ĠMIT
22: ĠCS
23: AIL
24: Ġch
25: un
26: wei
27: @
28: cs
29: ail
30: .mit
31: .edu
32: Ċ
33: ĠĠĠĠĠĠĠĠĠĠĠ
34: ĠAB
35: STRACT
36: Ċ
37: ĠĠĠĠĠĠĠĠĠĠĠ
38: ĠAnna
39: ĠPav
40: len
41: ko
42: ĠMicrosoft
43: Ġann
44: apa
45: @m
46: icrosoft
47: .com
48: Ċ
49: ĠĠĠĠĠĠĠĠĠĠĠ
50: ĠMatte
51: o
52: ĠInter
53: land
54: i
55: ĠMicrosoft
56: Ġmaint
57: er
58: l
59: @m
60: icrosoft
61: .com
62: Ċ
63: ĠĠĠĠĠĠĠĠĠĠĠ
64: ĠBrandon
65: ĠHay
66: nes
67: ĠMicrosoft
68: Ġbr
69: hay
70: nes
71: @m
72: icrosoft
73: .com
74: Ċ
75: ĠĠĠĠĠĠĠĠĠĠĠ
76: ĠThis
77: Ġpaper
78: Ġevaluates
79: Ġthe
80: Ġsuitability
81: Ġof
82: ĠApache
83: ĠArrow
84: ,
85: ĠPar
86: quet
87: ,
88: Ġand
89: ĠOR
90: C
91: Ġas
92: Ġformats
93: Ġfor
94: Ġsub
95: sum
96: ption
97: Ġin
98: Ġ

In [3]:
import torch

att_layers = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 16 and val >= 3:
                count += 1
                
    att_layers.append([f"Layer {layer}", count])
    # print(f"Layer: {layer} Count: {count}")
    
sorted_att_layer = sorted(att_layers, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att_layer[:10]

# Print the top 10
for item in top_10_att:
    print(item)

['Layer 21', 63]
['Layer 19', 60]
['Layer 22', 56]
['Layer 20', 55]
['Layer 23', 54]
['Layer 17', 53]
['Layer 24', 53]
['Layer 30', 53]
['Layer 15', 51]
['Layer 10', 48]


In [4]:
from src.attentivetrim.attention.attention_profiling import print_tokens_with_attention_head

print_tokens_with_attention_head(attention, tokens, 10, 31)

ImportError: MagickWand shared library not found.
You probably had not installed ImageMagick library.
Try to install:
  brew install freetype imagemagick

In [6]:
torch.sum(sum, dim=1)

tensor([32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        31.9844, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 31.9844, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000, 32.0000,
        32.0000, 32.0000, 32.0000, 32.00

In [7]:
import torch

att = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    multi_head = torch.sum(attention[layer], dim=(0))
    for head in range(0, 31):
        sum = multi_head[head]
        topk = torch.topk(sum[-10:], 30)
        topk.indices
        #iterate each element in the indices
        count = 0
        for index in topk.indices:
            
            for i in range(0, 30):
                val = index[i]
                if val <= 16 and val >= 3:
                    count += 1
        att.append([f"Layer: {layer}, Head: {head}", count])
        
        # print(f"Layer: {layer}, Head: {head}, Count: {count}")
sorted_att = sorted(att, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att[:10]

# Print the top 10
for item in top_10_att:
    print(item)


['Layer: 13, Head: 27', 90]
['Layer: 19, Head: 1', 90]
['Layer: 19, Head: 19', 90]
['Layer: 17, Head: 8', 88]
['Layer: 17, Head: 9', 88]
['Layer: 17, Head: 10', 88]
['Layer: 15, Head: 29', 85]
['Layer: 30, Head: 29', 83]
['Layer: 13, Head: 26', 82]
['Layer: 15, Head: 17', 82]


In [9]:
attention[0].shape


torch.Size([1, 32, 114, 114])

In [18]:
# print token with indices started from 0, add attention score as separate column
print("Token idx, token string, Attention Score for layer 13 and head 27")
question_att = torch.sum(attention[13][0][27][-10:], dim=0)
for i in range(0, 102):
    print(f"{i}: {tokens[i]}, {question_att[i]}")

Token idx, token string, Attention Score for layer 13 and head 27
0: <|begin_of_text|>, 7.7265625
1: Context, 0.056854248046875
2: :, 0.05218505859375
3: ĠA, 0.0308685302734375
4: ĠDeep, 0.0288848876953125
5: ĠDive, 0.054290771484375
6: Ġinto, 0.0928955078125
7: ĠCommon, 0.072265625
8: ĠOpen, 0.046539306640625
9: ĠFormats, 0.125732421875
10: Ġfor, 0.12646484375
11: ĠAnaly, 0.00954437255859375
12: tical, 0.00746917724609375
13: ĠDB, 0.0079345703125
14: MS, 0.0154571533203125
15: s, 0.023040771484375
16: Ċ, 0.396240234375
17: ĠĠĠĠĠĠĠĠĠĠĠ, 0.039703369140625
18: ĠChun, 0.0092010498046875
19: wei, 0.00494384765625
20: ĠLiu, 0.006404876708984375
21: ĠMIT, 0.0011034011840820312
22: ĠCS, 0.00022995471954345703
23: AIL, 0.00024378299713134766
24: Ġch, 0.0003790855407714844
25: un, 0.00019598007202148438
26: wei, 5.6624412536621094e-05
27: @, 0.000377655029296875
28: cs, 0.0001220703125
29: ail, 0.00016570091247558594
30: .mit, 0.0004622936248779297
31: .edu, 0.0016002655029296875
32: Ċ, 0.03576

In [58]:
i = 1
j = 0
print(f"context: {i} question: {j}")
        
# load the attention weights from the file
with open(f'../data/tensor/context{i}-question{j}_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open(f'../data/tensor/context{i}-question{j}_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")
for i in range(0, len(tokens)):
    print(f"{i}: {tokens[i]}")

context: 1 question: 0
Attention shape: 32
tokens length: 261
0: <|begin_of_text|>
1: Context
2: :
3: ĠA
4: ĠDECL
5: AR
6: ATIVE
7: ĠSYSTEM
8: ĠFOR
9: ĠOPT
10: IM
11: IZ
12: ING
13: ĠAI
14: ĠWORK
15: LOAD
16: S
17: Ċ
18: ĠĠĠĠĠĠĠĠĠĠĠ
19: ĠA
20: ĠPRE
21: PRINT
22: Ċ
23: ĠĠĠĠĠĠĠĠĠĠĠ
24: ĠChun
25: wei
26: ĠLiu
27: *Ċ
28: ĠĠĠĠĠĠĠĠĠĠĠ
29: Ġ,
30: ĠMatthew
31: ĠRusso
32: *Ċ
33: ĠĠĠĠĠĠĠĠĠĠĠ
34: Ġ,
35: ĠMichael
36: ĠCaf
37: arella
38: ,Ċ
39: ĠĠĠĠĠĠĠĠĠĠĠ
40: ĠLei
41: ĠCao
42: âĢł
43: Ċ
44: ĠĠĠĠĠĠĠĠĠĠĠ
45: Ġ,
46: ĠPeter
47: ĠB
48: aille
49: ĠChen
50: ,
51: ĠZ
52: ui
53: ĠChen
54: ,
55: ĠMichael
56: ĠFranklin
57: âĢ¡
58: Ċ
59: ĠĠĠĠĠĠĠĠĠĠĠ
60: Ġ,Ċ
61: ĠĠĠĠĠĠĠĠĠĠĠ
62: ĠTim
63: ĠKr
64: aska
65: ,
66: ĠSamuel
67: ĠMadden
68: ,
69: ĠGer
70: ardo
71: ĠVit
72: agli
73: ano
74: Ċ
75: ĠĠĠĠĠĠĠĠĠĠĠ
76: ĠMIT
77: ,
78: ĠâĢł
79: University
80: Ġof
81: ĠArizona
82: ,
83: ĠâĢ
84: ¡
85: University
86: Ġof
87: ĠChicago
88: Ċ
89: ĠĠĠĠĠĠĠĠĠĠĠ
90: Ġch
91: un
92: wei
93: @
94: mit
95: .edu
96: ,
97: Ġm
98: dr
99: us
100

In [59]:
import torch

att_layers = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 18 and val >= 3:
                count += 1
                
    att_layers.append([f"Layer {layer}", count])
    # print(f"Layer: {layer} Count: {count}")
    
sorted_att_layer = sorted(att_layers, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att_layer[:10]

# Print the top 10
for item in top_10_att:
    print(item)

['Layer 27', 72]
['Layer 30', 70]
['Layer 24', 69]
['Layer 26', 66]
['Layer 22', 60]
['Layer 25', 58]
['Layer 20', 56]
['Layer 21', 55]
['Layer 28', 52]
['Layer 16', 51]


In [51]:
import torch

att = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    multi_head = torch.sum(attention[layer], dim=(0))
    for head in range(0, 31):
        sum = multi_head[head]
        topk = torch.topk(sum[-10:], 30)
        topk.indices
        count = 0
        #iterate each element in the indices
        for index in topk.indices:
            
            for i in range(0, 30):
                val = index[i]
                if val <= 18 and val >= 3:
                    count += 1
        att.append([f"Layer: {layer}, Head: {head}", count])
        
        # print(f"Layer: {layer}, Head: {head}, Count: {count}")
sorted_att = sorted(att, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att[:10]

# Print the top 10
for item in top_10_att:
    print(item)

['Layer: 26, Head: 7', 124]
['Layer: 23, Head: 5', 103]
['Layer: 30, Head: 21', 102]
['Layer: 19, Head: 1', 98]
['Layer: 27, Head: 6', 98]
['Layer: 27, Head: 4', 95]
['Layer: 17, Head: 8', 93]
['Layer: 26, Head: 10', 93]
['Layer: 15, Head: 16', 92]
['Layer: 17, Head: 10', 91]


In [52]:
i = 2
j = 0
print(f"context: {i} question: {j}")
        
# load the attention weights from the file
with open(f'../data/tensor/context{i}-question{j}_attention.pkl', 'rb') as f:
    attention = pickle.load(f)
    
# load the tokens from the file
with open(f'../data/tensor/context{i}-question{j}_tokens.pkl', 'rb') as f:
    tokens = pickle.load(f)
    
print(f"Attention shape: {len(attention)}")
print(f"tokens length: {len(tokens)}")
for i in range(0, len(tokens)):
    print(f"{i}: {tokens[i]}")

context: 2 question: 0
Attention shape: 32
tokens length: 272
0: <|begin_of_text|>
1: Context
2: :
3: ĠDetect
4: ing
5: ĠAI
6: -
7: Generated
8: ĠText
9: :
10: ĠFactors
11: ĠInflu
12: encing
13: Ċ
14: Detect
15: ability
16: Ġwith
17: ĠCurrent
18: ĠMethods
19: Ċ
20: K
21: ath
22: leen
23: ĠC
24: .
25: ĠĊ
26: ĠĠĠĠĠĠĠĠĠĠĠĠ
27: ĠFraser
28: Ċ
29: k
30: ath
31: leen
32: .fr
33: aser
34: @n
35: rc
36: -cn
37: rc
38: .gc
39: .ca
40: Ċ
41: Hillary
42: ĠDaw
43: kins
44: Ċ
45: hill
46: ary
47: .d
48: aw
49: kins
50: @n
51: rc
52: -cn
53: rc
54: .gc
55: .ca
56: Ċ
57: S
58: vet
59: l
60: ana
61: ĠĊ
62: ĠĠĠĠĠĠĠĠĠĠĠĠ
63: ĠKir
64: itchen
65: ko
66: Ċ
67: sv
68: et
69: l
70: ana
71: .k
72: ir
73: itchen
74: ko
75: @n
76: rc
77: -cn
78: rc
79: .gc
80: .ca
81: Ċ
82: National
83: ĠResearch
84: ĠCouncil
85: ĠCanada
86: Ċ
87: 120
88: 0
89: ĠMontreal
90: ĠRoad
91: ,
92: ĠĊ
93: ĠĠĠĠĠĠĠĠĠĠĠĠ
94: ĠOttawa
95: ,
96: ĠCanada
97: Ċ
98: Abstract
99: Ċ
100: Large
101: Ġlanguage
102: Ġmodels
103: Ġ(
104: LL
105: Ms
10

In [53]:
import torch

att_layers = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    sum = torch.sum(attention[layer], dim=(0, 1))
    topk = torch.topk(sum[-10:], 30)
    topk.indices
    count = 0
    #iterate each element in the indices
    for index in topk.indices:
        for i in range(0, 30):
            val = index[i]
            if val <= 18 and val >= 3:
                count += 1
                
    att_layers.append([f"Layer {layer}", count])
    # print(f"Layer: {layer} Count: {count}")
    
sorted_att_layer = sorted(att_layers, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att_layer[:10]

# Print the top 10
for item in top_10_att:
    print(item)

['Layer 27', 78]
['Layer 30', 64]
['Layer 24', 62]
['Layer 16', 60]
['Layer 20', 59]
['Layer 21', 59]
['Layer 22', 57]
['Layer 25', 57]
['Layer 26', 56]
['Layer 19', 53]


In [54]:
import torch

att = []
for layer in range(0, 31):
    # print(f"Layer: {layer}")
    multi_head = torch.sum(attention[layer], dim=(0))
    for head in range(0, 31):
        sum = multi_head[head]
        topk = torch.topk(sum[-10:], 30)
        topk.indices
        count = 0
        #iterate each element in the indices
        for index in topk.indices:
            
            for i in range(0, 30):
                val = index[i]
                if val <= 18 and val >= 3:
                    count += 1
        att.append([f"Layer: {layer}, Head: {head}", count])
        
        # print(f"Layer: {layer}, Head: {head}, Count: {count}")
sorted_att = sorted(att, key=lambda x: x[1], reverse=True)

# Pick the top 10
top_10_att = sorted_att[:10]

# Print the top 10
for item in top_10_att:
    print(item)

['Layer: 15, Head: 29', 108]
['Layer: 19, Head: 1', 101]
['Layer: 13, Head: 26', 99]
['Layer: 13, Head: 27', 98]
['Layer: 24, Head: 17', 98]
['Layer: 27, Head: 5', 96]
['Layer: 9, Head: 27', 94]
['Layer: 27, Head: 4', 94]
['Layer: 30, Head: 29', 91]
['Layer: 16, Head: 0', 90]
